# 🧠 HA-NUN Grandmaster Training on Google Colab

This notebook trains the **210M Teacher → 21M Student** distillation pipeline on Google Colab with enterprise GPUs.

## Setup Steps
1. Mount Google Drive for persistent storage
2. Clone the repository
3. Install dependencies
4. Run grandmaster distillation training
5. Commit and push trained weights back to GitHub

In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
!ls -la /content/drive/MyDrive/ | head -20

In [ ]:
# STEP 2: Navigate to Drive and clone repository
%cd /content/drive/MyDrive/

# IMPORTANT: Replace with your GitHub username
GITHUB_USERNAME = "YOUR_GITHUB_USERNAME"
REPO_NAME = "trading-bot-HA-NUN"

# Clone if not already present
import os
if not os.path.exists(REPO_NAME):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

%cd {REPO_NAME}
!git pull origin main
!ls -la

In [ ]:
# STEP 3: Install system dependencies
!pip install -r requirements.txt

# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Also check for MPS (Apple Silicon) if running on Mac
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("MPS (Apple Silicon) available")
else:
    print("MPS not available")

In [ ]:
# STEP 4: Run Grandmaster Distillation Training (210M → 21M)

# Configuration for Colab training
TICKER = "SPY"           # Ticker to train on
PPO_TIMESTEPS = 1000000   # 1M timesteps for deep learning
EPOCHS = 50               # Transformer/LSTM epochs
DEVICE = "auto"           # auto-detect GPU

print("=" * 70)
print("  🧠 GRANDMASTER DISTILLATION TRAINING")
print("=" * 70)
print(f"Ticker: {TICKER}")
print(f"PPO Timesteps: {PPO_TIMESTEPS:,}")
print(f"Epochs: {EPOCHS}")
print(f"Device: {DEVICE}")
print("=" * 70)

# Run the training pipeline
!python main.py --mode advanced-train \
    --ticker {TICKER} \
    --ppo-timesteps {PPO_TIMESTEPS} \
    --epochs {EPOCHS} \
    --device {DEVICE} \
    --train-start 2020-01-01 \
    --train-end 2024-12-01 \
    --use-synthetic

In [ ]:
# STEP 5: Verify trained models exist
!ls -lah models/
!ls -lah models/checkpoints/
!ls -lah models/backups/

# Show training history
import json
try:
    with open('models/training_history.json', 'r') as f:
        history = json.load(f)
    print("\nTraining History:")
    print(f"Models trained: {history.get('models_trained', [])}")
    print(f"Data: {history.get('data', {})}")
    print(f"Metrics: {history.get('metrics', {})}")
except FileNotFoundError:
    print("No training history found yet.")

In [ ]:
# STEP 6: Commit and push trained weights to GitHub

!git config --global user.email "colab@ha-nun.ai"
!git config --global user.name "HA-NUN Colab Training"

# Stage all model files
!git add models/*.pth models/*.h5 models/*.json models/checkpoints/* models/backups/* models/training_history.json || true

# Check status
!git status --short

# Commit
!git commit -m "train: grandmaster distillation weights updated via Colab (210M→21M)" || echo "Nothing to commit"

# Push (you'll need to authenticate or use a token)
# Option A: If you have GITHUB_TOKEN set in environment
import os
token = os.getenv('GITHUB_TOKEN', '')
if token:
    !git push https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git main
else:
    print("\nTo push manually:")
    print(f"!git push origin main")
    print("\nOr use a token:")
    print(f"!git push https://<YOUR_TOKEN>@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git main")

In [ ]:
# OPTIONAL: Multi-Timeframe Training
# Train on different timeframes to make the AI universal

TIMEFRAMES = [
    {"name": "1min_scalper", "ticker": "SPY", "timeframe": "1min", "ppo_timesteps": 500000},
    {"name": "5min_scalper", "ticker": "SPY", "timeframe": "5min", "ppo_timesteps": 500000},
    {"name": "1h_swing", "ticker": "SPY", "timeframe": "1h", "ppo_timesteps": 300000},
    {"name": "4h_swing", "ticker": "SPY", "timeframe": "4h", "ppo_timesteps": 300000},
    {"name": "1d_position", "ticker": "SPY", "timeframe": "1d", "ppo_timesteps": 200000},
]

for tf_config in TIMEFRAMES:
    print(f"\n{'='*70}")
    print(f"  Training {tf_config['name']}...")
    print(f"{'='*70}")
    
    !python main.py --mode advanced-train \
        --ticker {tf_config['ticker']} \
        --timeframe {tf_config['timeframe']} \
        --ppo-timesteps {tf_config['ppo_timesteps']} \
        --epochs 30 \
        --device auto \
        --use-synthetic
    
    # Save with timeframe-specific name
    !cp models/transformer_model.pth models/transformer_model_{tf_config['name']}.pth
    !cp models/lstm_model.h5 models/lstm_model_{tf_config['name']}.h5
    !cp ppo_trader.zip ppo_trader_{tf_config['name']}.zip

In [ ]:
# OPTIONAL: LSE Training (London Stock Exchange)
# Train on LSE instruments for UK market trading

LSE_TICKERS = [
    # FTSE 100 ETFs and major UK stocks
    "ISF",   # iShares Core FTSE 100 UCITS ETF
    # "VOD",  # Vodafone
    # "BP",   # BP
    # "HSBA", # HSBC
]

for lse_ticker in LSE_TICKERS:
    print(f"\n{'='*70}")
    print(f"  Training LSE: {lse_ticker}")
    print(f"{'='*70}")
    
    !python main.py --mode advanced-train \
        --ticker {lse_ticker} \
        --lse \
        --timeframe 1h \
        --ppo-timesteps 300000 \
        --epochs 30 \
        --device auto \
        --use-synthetic
    
    # Save with LSE prefix
    !cp models/transformer_model.pth models/transformer_model_LSE_{lse_ticker}.pth
    !cp models/lstm_model.h5 models/lstm_model_LSE_{lse_ticker}.h5
    !cp ppo_trader.zip ppo_trader_LSE_{lse_ticker}.zip
    
    print(f"✅ LSE {lse_ticker} training complete")

In [ ]:
# FINAL: Push all new models to GitHub
!git add models/*.pth models/*.h5 models/*.zip models/*.json || true
!git add backtest_results/*.json || true
!git status --short
!git commit -m "feat: multi-timeframe + LSE grandmaster training complete" || echo "Nothing to commit"
!git push origin main || echo "Push manually if needed"